### Dataset Connection

In [47]:

import json
from pathlib import Path
import pandas as pd

# Locate project root
project_root = Path.cwd()

if not (project_root / "config/data_paths.json").exists():
    project_root = project_root.parent

# Load dataset configuration
with open(project_root / "config/data_paths.json") as f:
    config = json.load(f)

dataset_root = Path(config["dataset_root"])

print("Dataset found:", dataset_root.is_dir())
print("Dataset location:", dataset_root)


Dataset found: True
Dataset location: C:\Users\WINSYS\Desktop\Research project\Datasets\MXD\MXD Raw Data


### Dataset Structure Overview

In [48]:

# Find the main task folders (Composite and Wire Harness)
task_folders = sorted(
    folder for folder in dataset_root.iterdir()
    if folder.is_dir()
)

# Initialize the total CSV file counter
total_csvs = 0

# Loop through each task folder
for task in task_folders:

    # Find participant folders (P1, P2, P3, ...)
    participants = sorted(
        folder for folder in task.iterdir()
        if folder.is_dir() and folder.name.startswith("P")
    )

    # Find CSV files inside each participant's Processed Files folder
    csv_files = [
        file
        for participant in participants
        for file in (participant / "Processed Files").rglob("*.csv")
    ]

    # Count the processed CSV files
    total_csvs += len(csv_files)

    # Display results for each task
    print(f"Task: {task.name}")
    print(f"Participants: {len(participants)}")
    print(f"Processed CSV files: {len(csv_files)}")
    print("-" * 40)


print(f"Total processed CSV files: {total_csvs}")


Task: Composite
Participants: 43
Processed CSV files: 252
----------------------------------------
Task: Wire Harness
Participants: 43
Processed CSV files: 217
----------------------------------------
Total processed CSV files: 469



### CSV File Organization

Examine the number of CSV files per participant and identify the available experimental sessions.


In [49]:

from collections import Counter

# Examine each task
for task in task_folders:

    # Find participant folders
    participants = sorted(
        folder for folder in task.iterdir()
        if folder.is_dir() and folder.name.startswith("P")
    )

    # Count processed CSV files for each participant
    file_counts = {
        p.name: len(list((p / "Processed Files").glob("*.csv")))
        for p in participants
    }

    # Summarize how many participants have each file count
    count_summary = Counter(file_counts.values())

    print(f"\nTask: {task.name}")
    print("CSV count distribution:", dict(sorted(count_summary.items())))

    # Identify participants with fewer than 7 CSV files
    incomplete = [p for p, count in file_counts.items() if count < 7]
    print("Participants with fewer than 7 files:", incomplete)


    example_folder = task / "P1" / "Processed Files"

    print("P1 CSV files:")
    for file in sorted(example_folder.glob("*.csv")):
        print(" -", file.name)



Task: Composite
CSV count distribution: {0: 7, 7: 36}
Participants with fewer than 7 files: ['P15', 'P23', 'P24', 'P25', 'P26', 'P35', 'P9']
P1 CSV files:
 - Final_Rest_P001C001S002.csv
 - Initial_Rest_P001C001S002.csv
 - Rep1_P001C001S002.csv
 - Rep2_P001C001S002.csv
 - Rep3_P001C001S002.csv
 - Rep4_P001C001S002.csv
 - Rep5_P001C001S002.csv

Task: Wire Harness
CSV count distribution: {0: 12, 7: 31}
Participants with fewer than 7 files: ['P10', 'P13', 'P18', 'P23', 'P24', 'P25', 'P26', 'P3', 'P31', 'P34', 'P35', 'P6']
P1 CSV files:
 - Final_Rest_P001Z001S003.csv
 - Initial_Rest_P001Z001S003.csv
 - Rep1_P001Z001S003.csv
 - Rep2_P001Z001S003.csv
 - Rep3_P001Z001S003.csv
 - Rep4_P001Z001S003.csv
 - Rep5_P001Z001S003.csv


### Inspect CSV file contenets to undersatd the structure

In [50]:

# Select the Composite P1 processed files folder
sample_folder = dataset_root / "Composite" / "P1" / "Processed Files"

# Find the Rep1 CSV file
sample_file = next(sample_folder.glob("Rep1_*.csv"))

# Load only the first 5 rows
sample_df = pd.read_csv(sample_file, nrows=10)


print("File:", sample_file.name)


print(f"\nTotal columns: {len(sample_df.columns)}")

for column in sample_df.columns:
    print(column)

sample_df.head(10)


File: Rep1_P001C001S002.csv

Total columns: 51
Time
HR_Processed
HR
HRV
RR
RRSQI
ECGSQI
ECG
acc_X
acc_Y
acc_Z
Temperature
IMU_1_ax_g_
IMU_1_ay_g_
IMU_1_az_g_
IMU_1_gx_dps_
IMU_1_gy_dps_
IMU_1_gz_dps_
IMU_2_ax_g_
IMU_2_ay_g_
IMU_2_az_g_
IMU_2_gx_dps_
IMU_2_gy_dps_
IMU_2_gz_dps_
IMU_3_ax_g_
IMU_3_ay_g_
IMU_3_az_g_
IMU_3_gx_dps_
IMU_3_gy_dps_
IMU_3_gz_dps_
IMU_4_ax_g_
IMU_4_ay_g_
IMU_4_az_g_
IMU_4_gx_dps_
IMU_4_gy_dps_
IMU_4_gz_dps_
IMU_5_ax_g_
IMU_5_ay_g_
IMU_5_az_g_
IMU_5_gx_dps_
IMU_5_gy_dps_
IMU_5_gz_dps_
Physical Fatigue - Initial
Physical Fatigue - Final
Mental Fatigue - Initial
Mental Fatigue - Final
Performance rating
Age
Weight
Height
Weights added


,Time,HR_Processed,HR,HRV,RR,RRSQI,ECGSQI,ECG,acc_X,acc_Y,...,IMU_5_gz_dps_,Physical Fatigue - Initial,Physical Fatigue - Final,Mental Fatigue - Initial,Mental Fatigue - Final,Performance rating,Age,Weight,Height,Weights added
0,522500,102.5,130.466306,22.402814,15.9,0.496,0.811,0.823492,0.066127,0.043541,...,14.436704,1,3,1,1,92.91,24,216,190,35
1,523000,104.7,130.434573,22.376775,15.9,0.496,0.811,-0.650494,-1.324813,0.082063,...,1.233281,1,3,1,1,92.91,24,216,190,35
2,523500,108.2,130.476463,22.351200,15.9,0.496,0.804,-3.303531,-0.625702,-0.055543,...,13.288418,1,3,1,1,92.91,24,216,190,35
3,524000,108.2,130.314845,22.326121,15.9,0.496,0.804,1.942350,-0.054765,-0.012957,...,-34.480159,1,3,1,1,92.91,24,216,190,35
4,524500,108.2,129.849866,22.301569,15.9,0.496,0.804,1.410120,-0.182991,-0.003705,...,-5.812640,1,3,1,1,92.91,24,216,190,35
5,525000,108.2,129.851213,22.277574,15.9,0.496,0.804,0.087020,-0.492419,-0.106889,...,-16.733941,1,3,1,1,92.91,24,216,190,35
6,525500,108.2,129.479952,22.254168,15.9,0.496,0.804,-0.158387,-0.508931,0.031095,...,-6.454106,1,3,1,1,92.91,24,216,190,35
7,526000,108.2,129.405652,22.231382,15.9,0.496,0.804,-0.059286,-0.466193,-0.123889,...,3.563215,1,3,1,1,92.91,24,216,190,35
8,526500,108.2,129.765739,22.209245,15.9,0.496,0.804,-0.068347,-0.497865,0.029170,...,-16.625290,1,3,1,1,92.91,24,216,190,35
9,527000,108.2,129.916332,22.187790,15.9,0.496,0.804,0.017942,-0.477887,-0.037307,...,5.428093,1,3,1,1,92.91,24,216,190,35



### Column Consistency

Compare the columns in Composite and Wire Harness to determine whether their processed data can be combined using a common structure.


In [51]:

# Select a sample CSV from Wire Harness P1
wire_folder = dataset_root / "Wire Harness" / "P1" / "Processed Files"
wire_file = next(wire_folder.glob("Rep1_*.csv"))

# Read only column names without loading sensor data
composite_columns = pd.read_csv(sample_file, nrows=0).columns.tolist()
wire_columns = pd.read_csv(wire_file, nrows=0).columns.tolist()

# Compare the columns in both tasks
print("Composite columns:", len(composite_columns))
print("Wire Harness columns:", len(wire_columns))
print("Same columns and order:", composite_columns == wire_columns)

# Identify columns unique to each task
print("\nOnly in Composite:", sorted(set(composite_columns) - set(wire_columns)))
print("Only in Wire Harness:", sorted(set(wire_columns) - set(composite_columns)))


Composite columns: 51
Wire Harness columns: 51
Same columns and order: True

Only in Composite: []
Only in Wire Harness: []


In [52]:

# Find all processed CSV files
processed_files = sorted(
    dataset_root.glob("*/P*/Processed Files/*.csv")
)

# Use the sample CSV columns as the reference
reference_columns = sample_df.columns.tolist()

# Store files with different columns or reading errors
mismatched_files = []
read_errors = []

# Check the column names in every processed CSV
for file in processed_files:
    try:
        columns = pd.read_csv(file, nrows=0).columns.tolist()

        # Identify files with different columns or column order
        if columns != reference_columns:
            mismatched_files.append(str(file.relative_to(dataset_root)))

    except Exception as error:
        read_errors.append((str(file.relative_to(dataset_root)), str(error)))


print("Total CSV files:", len(processed_files))
print("Files with different columns:", len(mismatched_files))
print("Files with reading errors:", len(read_errors))


print("\nDifferent columns:", mismatched_files[:10])
print("Reading errors:", read_errors[:10])


Total CSV files: 469
Files with different columns: 0
Files with reading errors: 0

Different columns: []
Reading errors: []



### Physical Fatigue Scores

Understand how MXD records fatigue scores across Initial Rest, Rep1–Rep5, and Final Rest.


In [53]:

# Define the physical fatigue columns
fatigue_columns = [
    "Physical Fatigue - Initial",
    "Physical Fatigue - Final"
]

# Define the experimental sessions in order
sessions = [
    "Initial_Rest",
    "Rep1",
    "Rep2",
    "Rep3",
    "Rep4",
    "Rep5",
    "Final_Rest"
]

# Store the fatigue scores
fatigue_results = []

# Read the fatigue scores from Composite P1
for session in sessions:

    # Find the corresponding CSV file
    file = next(sample_folder.glob(f"{session}_*.csv"))

    # Read only the first row of the fatigue columns
    data = pd.read_csv(file, usecols=fatigue_columns, nrows=1)

    # Store the session and its scores
    fatigue_results.append({
        "Session": session,
        "Initial Fatigue": data.iloc[0]["Physical Fatigue - Initial"],
        "Final Fatigue": data.iloc[0]["Physical Fatigue - Final"]
    })

# Convert the results into a DataFrame
fatigue_df = pd.DataFrame(fatigue_results)


display(fatigue_df)


,Session,Initial Fatigue,Final Fatigue
0,Initial_Rest,1,1
1,Rep1,1,3
2,Rep2,3,5
3,Rep3,5,6
4,Rep4,6,7
5,Rep5,7,7
6,Final_Rest,7,3



### Fatigue Score Consistency

Check whether initial and final fatigue scores remain consistent throughout each session and identify missing scores.


In [54]:

# Store the score consistency results
consistency_results = []

# Examine all seven sessions for Composite P1
for session in sessions:

    # Find the corresponding CSV file
    file = next(sample_folder.glob(f"{session}_*.csv"))

    # Read only the physical fatigue columns
    scores = pd.read_csv(file, usecols=fatigue_columns)

    # Count unique scores and missing values
    consistency_results.append({
        "Session": session,
        "Rows": len(scores),
        "Unique Initial Scores": scores[fatigue_columns[0]].nunique(),
        "Unique Final Scores": scores[fatigue_columns[1]].nunique(),
        "Missing Scores": scores.isna().sum().sum()
    })

consistency_df = pd.DataFrame(consistency_results)
display(consistency_df)

,Session,Rows,Unique Initial Scores,Unique Final Scores,Missing Scores
0,Initial_Rest,1036,1,1,0
1,Rep1,1029,1,1,0
2,Rep2,867,1,1,0
3,Rep3,826,1,1,0
4,Rep4,788,1,1,0
5,Rep5,697,1,1,0
6,Final_Rest,862,1,1,0


###  Dataset-Wide Fatigue Scores

In [55]:

# Store fatigue scores from all processed CSV files
score_records = []

# Examine each processed CSV file
for file in processed_files:

    # Read only the first row of the fatigue columns
    data = pd.read_csv(file, usecols=fatigue_columns, nrows=1)

    # Store task, participant, and fatigue scores
    score_records.append({
        "Task": file.parents[2].name,
        "Participant": file.parents[1].name,
        "Initial Fatigue": data.iloc[0]["Physical Fatigue - Initial"],
        "Final Fatigue": data.iloc[0]["Physical Fatigue - Final"]
    })

# Convert the results into a DataFrame
scores_df = pd.DataFrame(score_records)


print("Total files examined:", len(scores_df))

for column in fatigue_columns:
    result_column = (
        "Initial Fatigue" if "Initial" in column else "Final Fatigue"
    )

    print(f"\n{result_column}")
    print("Unique scores:", sorted(scores_df[result_column].dropna().unique()))
    print("Missing values:", scores_df[result_column].isna().sum())


Total files examined: 469

Initial Fatigue
Unique scores: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Missing values: 0

Final Fatigue
Unique scores: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Missing values: 0


### Fatigue Score Distribution

In [56]:

# Count the frequency of each initial fatigue score
initial_counts = scores_df["Initial Fatigue"].value_counts()

# Count the frequency of each final fatigue score
final_counts = scores_df["Final Fatigue"].value_counts()

# Combine the score frequencies into one table
score_distribution = pd.DataFrame({
    "Initial Count": initial_counts,
    "Final Count": final_counts
}).fillna(0).astype(int)

# Arrange scores from 0 to 10
score_distribution = score_distribution.reindex(
    range(11), fill_value=0
)

score_distribution.index.name = "Fatigue Score"
display(score_distribution)


,Initial Count,Final Count
Fatigue Score,,
0,44,31
1,58,52
2,55,49
3,58,61
4,47,52
5,71,71
6,49,56
7,35,40
8,33,37



### Fatigue Level Distribution


In [57]:

# Define the fatigue level categories
fatigue_levels = ["Low", "Moderate", "High"]

# Convert initial fatigue scores into three levels
initial_levels = pd.cut(
    scores_df["Initial Fatigue"],
    bins=[-1, 3, 6, 10],
    labels=fatigue_levels
)

# Convert final fatigue scores into three levels
final_levels = pd.cut(
    scores_df["Final Fatigue"],
    bins=[-1, 3, 6, 10],
    labels=fatigue_levels
)

# Count the frequency of each fatigue level
level_distribution = pd.DataFrame({
    "Initial Count": initial_levels.value_counts(),
    "Final Count": final_levels.value_counts()
})

# Arrange the fatigue levels in order
level_distribution = level_distribution.reindex(fatigue_levels)

display(level_distribution)


,Initial Count,Final Count
Low,215,193
Moderate,167,179
High,87,97



### Task-Wise Fatigue Distribution


In [58]:

# Add the fatigue level categories to the dataset
scores_df["Initial Level"] = initial_levels
scores_df["Final Level"] = final_levels

# Count initial fatigue levels for each task
initial_by_task = pd.crosstab(
    scores_df["Task"],
    scores_df["Initial Level"]
).reindex(columns=fatigue_levels, fill_value=0)

# Count final fatigue levels for each task
final_by_task = pd.crosstab(
    scores_df["Task"],
    scores_df["Final Level"]
).reindex(columns=fatigue_levels, fill_value=0)

# Combine both distributions into one table
task_distribution = pd.concat(
    {"Initial": initial_by_task, "Final": final_by_task},
    names=["Stage", "Task"]
)

display(task_distribution)


Low  Moderate  High
Stage   Task                             
Initial Composite     124        91    37
        Wire Harness   91        76    50
Final   Composite     114        97    41
        Wire Harness   79        82    56


### Session-Wise Fatigue Progression


In [59]:

# Extract the session name from each CSV filename
scores_df["Session"] = [
    file.name.split("_P")[0]
    for file in processed_files
]

# Define the experimental session order
session_order = [
    "Initial_Rest", "Rep1", "Rep2", "Rep3",
    "Rep4", "Rep5", "Final_Rest"
]

# Calculate average initial and final fatigue scores for each task and session
session_summary = (
    scores_df.groupby(["Task", "Session"])[
        ["Initial Fatigue", "Final Fatigue"]
    ]
    .mean()
    .round(2)
    .reset_index()
)

# Arrange sessions in experimental order
session_summary["Session"] = pd.Categorical(
    session_summary["Session"],
    categories=session_order,
    ordered=True
)

session_summary = session_summary.sort_values(
    ["Task", "Session"]
)

display(session_summary)


,Task,Session,Initial Fatigue,Final Fatigue
1,Composite,Initial_Rest,1.67,1.67
2,Composite,Rep1,1.67,3.22
3,Composite,Rep2,3.22,4.17
4,Composite,Rep3,4.17,4.81
5,Composite,Rep4,4.81,5.31
6,Composite,Rep5,5.31,5.53
0,Composite,Final_Rest,5.53,3.25
8,Wire Harness,Initial_Rest,1.71,1.71
9,Wire Harness,Rep1,1.71,3.87
10,Wire Harness,Rep2,3.87,4.61



### Fatigue Score Continuity


In [60]:

# Arrange the sessions in experimental order
continuity_df = scores_df.copy()

continuity_df["Session"] = pd.Categorical(
    continuity_df["Session"],
    categories=session_order,
    ordered=True
)

continuity_df = continuity_df.sort_values(
    ["Task", "Participant", "Session"]
)

# Get the previous session's final fatigue score
continuity_df["Previous Final"] = (
    continuity_df.groupby(["Task", "Participant"])["Final Fatigue"]
    .shift(1)
)

# Select sessions that have a previous session
transitions = continuity_df[
    continuity_df["Previous Final"].notna()
].copy()

# Check whether consecutive fatigue scores match
transitions["Matches"] = (
    transitions["Initial Fatigue"] == transitions["Previous Final"]
)

# Identify inconsistent sessions
mismatches = transitions[~transitions["Matches"]]

print("Transitions checked:", len(transitions))
print("Matching transitions:", transitions["Matches"].sum())
print("Mismatched transitions:", len(mismatches))

display(mismatches[
    ["Task", "Participant", "Session",
     "Previous Final", "Initial Fatigue"]
].head(10))


Transitions checked: 402
Matching transitions: 402
Mismatched transitions: 0


,Task,Participant,Session,Previous Final,Initial Fatigue



### Missing Value Analysis


In [61]:

# Create counters for missing values across all columns
missing_counts = pd.Series(0, index=reference_columns, dtype="int64")
sampled_rows = 0

# Read up to 500 rows from each processed CSV file
for file in processed_files:
    data = pd.read_csv(file, nrows=500)

    # Count missing values in each column
    missing_counts += data.isna().sum()
    sampled_rows += len(data)

# Calculate the percentage of missing values
missing_summary = pd.DataFrame({
    "Missing Values": missing_counts,
    "Missing (%)": (missing_counts / sampled_rows * 100).round(2)
})

# Sort columns from most missing values to least
missing_summary = missing_summary.sort_values(
    "Missing Values", ascending=False
)

print("CSV files checked:", len(processed_files))
print("Rows sampled:", sampled_rows)
print("Columns with missing values:", (missing_counts > 0).sum())

display(missing_summary.head(20))


CSV files checked: 469
Rows sampled: 223382
Columns with missing values: 0


,Missing Values,Missing (%)
Time,0,0.0
HR_Processed,0,0.0
HR,0,0.0
HRV,0,0.0
RR,0,0.0
RRSQI,0,0.0
ECGSQI,0,0.0
ECG,0,0.0
acc_X,0,0.0
acc_Y,0,0.0



### Sensor Value Range Analysis


In [62]:

# Select important physiological and motion sensor columns
sensor_columns = [
    "HR_Processed", "HR", "HRV",
    "Temperature", "acc_X", "acc_Y", "acc_Z"
]

# Collect a sample from every processed CSV file
sensor_samples = []

for file in processed_files:
    data = pd.read_csv(
        file,
        usecols=sensor_columns,
        nrows=500
    )

    sensor_samples.append(data)

# Combine the sampled sensor readings
sensor_df = pd.concat(sensor_samples, ignore_index=True)

# Convert sensor columns to numeric values
for column in sensor_columns:
    sensor_df[column] = pd.to_numeric(
        sensor_df[column],
        errors="coerce"
    )

# Calculate sensor statistics
sensor_summary = sensor_df[sensor_columns].agg(
    ["min", "max", "mean", "std"]
).T.round(2)

# Count zero values and non-numeric or missing values
sensor_summary["Zero Count"] = (
    sensor_df[sensor_columns] == 0
).sum()

sensor_summary["Missing/Invalid Count"] = (
    sensor_df[sensor_columns].isna().sum()
)

print("Sensor readings sampled:", len(sensor_df))

display(sensor_summary)


Sensor readings sampled: 223382


,min,max,mean,std,Zero Count,Missing/Invalid Count
HR_Processed,-3.000000e+00,2.324000e+02,86.11,4.243000e+01,0,0
HR,-8.121669e+04,1.256160e+03,62.71,1.098970e+03,0,0
HRV,-1.570748e+11,9.950492e+08,-83942578.91,2.755277e+09,0,0
Temperature,1.973000e+01,3.904000e+01,34.18,1.800000e+00,0,0
acc_X,-5.718600e+02,5.948000e+02,0.70,1.719000e+01,0,0
acc_Y,-5.952300e+02,5.932600e+02,0.32,9.220000e+00,0,0
acc_Z,-4.183339e+04,9.305385e+04,1.04,3.113100e+02,0,0



### Sensor Outlier Investigation


In [63]:

# Calculate percentiles to understand typical sensor values
sensor_percentiles = sensor_df[sensor_columns].quantile(
    [0.01, 0.50, 0.99]
).T

# Rename the percentile columns
sensor_percentiles.columns = [
    "1st Percentile", "Median", "99th Percentile"
]

sensor_percentiles = sensor_percentiles.round(2)

display(sensor_percentiles)

# Examine potentially invalid heart-rate readings
hr_checks = pd.DataFrame({
    "Nonpositive (<= 0)": [
        (sensor_df[column] <= 0).sum()
        for column in ["HR_Processed", "HR"]
    ],
    "Above 220": [
        (sensor_df[column] > 220).sum()
        for column in ["HR_Processed", "HR"]
    ]
}, index=["HR_Processed", "HR"])

# Count negative HRV readings separately
negative_hrv = (sensor_df["HRV"] < 0).sum()

display(hr_checks)
print("Negative HRV readings:", negative_hrv)
print("Total sampled rows:", len(sensor_df))


,1st Percentile,Median,99th Percentile
HR_Processed,-3.00,98.50,151.80
HR,18.78,99.94,142.63
HRV,-183307.39,55.64,195024.93
Temperature,28.65,34.32,38.06
acc_X,-57.72,-0.45,68.19
acc_Y,-25.40,0.02,27.13
acc_Z,-23.45,-0.04,25.21


,Nonpositive (<= 0),Above 220
HR_Processed,36386,2
HR,835,10


Negative HRV readings: 7872
Total sampled rows: 223382



### Invalid Heart Rate Value Investigation


In [64]:

# Select nonpositive processed heart-rate readings
invalid_hr = sensor_df.loc[
    sensor_df["HR_Processed"] <= 0,
    "HR_Processed"
]

# Count how often each invalid value appears
invalid_hr_summary = (
    invalid_hr.value_counts()
    .sort_index()
    .rename_axis("HR_Processed Value")
    .reset_index(name="Count")
)

# Calculate each value's percentage of sampled rows
invalid_hr_summary["Percentage (%)"] = (
    invalid_hr_summary["Count"] / len(sensor_df) * 100
).round(2)

print("Distinct nonpositive values:", len(invalid_hr_summary))

display(invalid_hr_summary)


Distinct nonpositive values: 2


,HR_Processed Value,Count,Percentage (%)
0,-3.0,8075,3.61
1,-2.0,28311,12.67



### Heart Rate Quality Investigation


In [65]:

# Select heart-rate and signal-quality columns
quality_columns = [
    "HR_Processed", "HR", "RRSQI", "ECGSQI"
]

# Read up to 500 rows from each processed CSV file
quality_samples = []

for file in processed_files:
    data = pd.read_csv(
        file,
        usecols=quality_columns,
        nrows=500
    )

    quality_samples.append(data)

# Combine the sampled readings
quality_df = pd.concat(quality_samples, ignore_index=True)

# Separate special codes from other processed HR readings
quality_df["HR Group"] = "Other"

quality_df.loc[
    quality_df["HR_Processed"] == -2,
    "HR Group"
] = "-2"

quality_df.loc[
    quality_df["HR_Processed"] == -3,
    "HR Group"
] = "-3"

# Compare raw HR and signal-quality measurements
quality_summary = quality_df.groupby("HR Group").agg(
    Count=("HR", "size"),
    Median_Raw_HR=("HR", "median"),
    Median_RRSQI=("RRSQI", "median"),
    Median_ECGSQI=("ECGSQI", "median")
).round(2)

# Arrange the groups for comparison
quality_summary = quality_summary.reindex(
    ["-3", "-2", "Other"]
)

display(quality_summary)


,Count,Median_Raw_HR,Median_RRSQI,Median_ECGSQI
HR Group,,,,
-3,8075,96.06,0.0,0.00
-2,28311,96.17,0.0,0.00
Other,186996,100.68,0.0,0.72



### ECG Signal Quality Distribution


In [66]:

# Identify readings with zero ECG signal quality
quality_df["ECGSQI Is Zero"] = (
    quality_df["ECGSQI"] == 0
)

# Count zero-quality readings for each HR group
ecg_summary = quality_df.groupby("HR Group").agg(
    Total_Readings=("ECGSQI", "size"),
    Zero_ECGSQI_Count=("ECGSQI Is Zero", "sum")
)

# Calculate the percentage of zero-quality readings
ecg_summary["Zero_ECGSQI_Percentage"] = (
    ecg_summary["Zero_ECGSQI_Count"]
    / ecg_summary["Total_Readings"]
    * 100
).round(2)

# Arrange the groups in order
ecg_summary = ecg_summary.reindex(
    ["-3", "-2", "Other"]
)

display(ecg_summary)


,Total_Readings,Zero_ECGSQI_Count,Zero_ECGSQI_Percentage
HR Group,,,
-3,8075,8075,100.0
-2,28311,28311,100.0
Other,186996,0,0.0



### Invalid Heart Rate by Task


In [67]:

# Count special heart-rate codes in each task
task_quality_records = []

for file in processed_files:
    data = pd.read_csv(
        file,
        usecols=["HR_Processed"],
        nrows=500
    )

    task_quality_records.append({
        "Task": file.relative_to(dataset_root).parts[0],
        "Sampled Rows": len(data),
        "Code -2": (data["HR_Processed"] == -2).sum(),
        "Code -3": (data["HR_Processed"] == -3).sum()
    })

# Combine counts across sessions for each task
task_quality_df = pd.DataFrame(task_quality_records)

task_quality_summary = task_quality_df.groupby("Task").sum()

# Calculate the proportion of readings with special codes
task_quality_summary["Invalid HR (%)"] = (
    (
        task_quality_summary["Code -2"]
        + task_quality_summary["Code -3"]
    )
    / task_quality_summary["Sampled Rows"]
    * 100
).round(2)

display(task_quality_summary)


,Sampled Rows,Code -2,Code -3,Invalid HR (%)
Task,,,,
Composite,119063,15415,1393,14.12
Wire Harness,104319,12896,6682,18.77



### Participant-Wise Invalid Heart Rate Analysis


In [68]:

# Store invalid heart-rate statistics for each participant and task
participant_quality_records = []

for file in processed_files:
    # Identify the task and participant from the file path
    relative_parts = file.relative_to(dataset_root).parts
    task = relative_parts[0]
    participant = relative_parts[1]

    # Read up to 500 processed heart-rate readings
    data = pd.read_csv(
        file,
        usecols=["HR_Processed"],
        nrows=500
    )

    # Count readings containing the special codes
    invalid_count = data["HR_Processed"].isin([-2, -3]).sum()

    participant_quality_records.append({
        "Task": task,
        "Participant": participant,
        "Sampled Rows": len(data),
        "Invalid Readings": invalid_count
    })

# Combine session results for each participant within each task
participant_quality_df = pd.DataFrame(participant_quality_records)

participant_quality_summary = (
    participant_quality_df
    .groupby(["Task", "Participant"])[
        ["Sampled Rows", "Invalid Readings"]
    ]
    .sum()
)

# Calculate the invalid-reading percentage
participant_quality_summary["Invalid HR (%)"] = (
    participant_quality_summary["Invalid Readings"]
    / participant_quality_summary["Sampled Rows"]
    * 100
).round(2)

# Identify participants with the highest invalid-reading percentages
participant_quality_summary = participant_quality_summary.sort_values(
    "Invalid HR (%)",
    ascending=False
)

print("Participant-task combinations:", len(participant_quality_summary))
print(
    "Combinations with over 50% invalid HR:",
    (participant_quality_summary["Invalid HR (%)"] > 50).sum()
)

display(participant_quality_summary.head(15))


Participant-task combinations: 67
Combinations with over 50% invalid HR: 8


Sampled Rows  Invalid Readings  Invalid HR (%)
Task         Participant                                                
Composite    P36                  3500              2663           76.09
             P31                  2446              1804           73.75
             P28                  3096              2136           68.99
Wire Harness P14                  3500              2392           68.34
Composite    P41                  3411              2259           66.23
Wire Harness P28                  3404              2034           59.75
             P19                  3258              1809           55.52
Composite    P30                  2803              1404           50.09
Wire Harness P37                  3500              1560           44.57
             P36                  3485              1509           43.30
Composite    P39                  3456              1452           42.01
Wire Harness P11                  3172              1074           33.86
             P16                  3285              1082           32.94
Composite    P43                  3500              1102           31.49
Wire Harness P27                  3199               983           30.73


## Session-Wise Invalid Heart Rate Analysis


In [69]:

# Store heart-rate quality statistics for each session
session_quality_records = []

for file in processed_files:
    # Identify the task and experimental session
    task = file.relative_to(dataset_root).parts[0]
    session = file.name.split("_P")[0]

    # Read up to 500 processed heart-rate readings
    data = pd.read_csv(
        file,
        usecols=["HR_Processed"],
        nrows=500
    )

    # Count readings with the special codes
    invalid_count = data["HR_Processed"].isin([-2, -3]).sum()

    session_quality_records.append({
        "Task": task,
        "Session": session,
        "Sampled Rows": len(data),
        "Invalid Readings": invalid_count
    })

# Combine results by task and session
session_quality_df = pd.DataFrame(session_quality_records)

session_quality_summary = (
    session_quality_df
    .groupby(["Task", "Session"])[
        ["Sampled Rows", "Invalid Readings"]
    ]
    .sum()
    .reset_index()
)

# Calculate invalid-reading percentages
session_quality_summary["Invalid HR (%)"] = (
    session_quality_summary["Invalid Readings"]
    / session_quality_summary["Sampled Rows"]
    * 100
).round(2)

# Arrange sessions in experimental order
session_quality_summary["Session"] = pd.Categorical(
    session_quality_summary["Session"],
    categories=session_order,
    ordered=True
)

session_quality_summary = session_quality_summary.sort_values(
    ["Task", "Session"]
)

display(session_quality_summary)


,Task,Session,Sampled Rows,Invalid Readings,Invalid HR (%)
1,Composite,Initial_Rest,13537,1391,10.28
2,Composite,Rep1,17369,2876,16.56
3,Composite,Rep2,17916,3296,18.40
4,Composite,Rep3,17877,2860,16.00
5,Composite,Rep4,17718,2334,13.17
6,Composite,Rep5,17629,2342,13.28
0,Composite,Final_Rest,17017,1709,10.04
8,Wire Harness,Initial_Rest,11905,797,6.69
9,Wire Harness,Rep1,15500,1510,9.74
10,Wire Harness,Rep2,15500,2790,18.00



### Full Dataset Heart Rate Quality Check


In [70]:

# Store complete heart-rate quality statistics
full_quality_records = []

# Process each CSV in memory-efficient chunks
for file in processed_files:
    task = file.relative_to(dataset_root).parts[0]
    session = file.name.split("_P")[0]

    total_rows = 0
    code_minus_2 = 0
    code_minus_3 = 0
    other_nonpositive = 0
    missing_values = 0

    # Read only the processed heart-rate column
    for chunk in pd.read_csv(
        file,
        usecols=["HR_Processed"],
        chunksize=50000
    ):
        values = pd.to_numeric(
            chunk["HR_Processed"],
            errors="coerce"
        )

        total_rows += len(values)
        code_minus_2 += (values == -2).sum()
        code_minus_3 += (values == -3).sum()

        # Identify any other nonpositive readings
        other_nonpositive += (
            (values <= 0) & ~values.isin([-2, -3])
        ).sum()

        missing_values += values.isna().sum()

    full_quality_records.append({
        "Task": task,
        "Session": session,
        "Total Rows": total_rows,
        "Code -2": code_minus_2,
        "Code -3": code_minus_3,
        "Other Nonpositive": other_nonpositive,
        "Missing/Non-numeric": missing_values
    })

# Combine statistics by task and session
full_quality_df = pd.DataFrame(full_quality_records)

full_quality_summary = (
    full_quality_df
    .groupby(["Task", "Session"])
    .sum()
    .reset_index()
)

# Calculate the percentage of special heart-rate codes
full_quality_summary["Invalid HR (%)"] = (
    (
        full_quality_summary["Code -2"]
        + full_quality_summary["Code -3"]
    )
    / full_quality_summary["Total Rows"]
    * 100
).round(2)

# Arrange sessions in experimental order
full_quality_summary["Session"] = pd.Categorical(
    full_quality_summary["Session"],
    categories=session_order,
    ordered=True
)

full_quality_summary = full_quality_summary.sort_values(
    ["Task", "Session"]
)

print("CSV files checked:", len(full_quality_records))
print("Total sensor rows:", full_quality_df["Total Rows"].sum())

display(full_quality_summary)


CSV files checked: 469
Total sensor rows: 363603


,Task,Session,Total Rows,Code -2,Code -3,Other Nonpositive,Missing/Non-numeric,Invalid HR (%)
1,Composite,Initial_Rest,18599,1533,93,0,0,8.74
2,Composite,Rep1,33764,5020,0,0,0,14.87
3,Composite,Rep2,28874,4436,26,0,0,15.45
4,Composite,Rep3,27320,2630,749,0,0,12.37
5,Composite,Rep4,25094,2598,151,0,0,10.95
6,Composite,Rep5,24164,2632,404,0,0,12.56
0,Composite,Final_Rest,23127,1643,366,0,0,8.69
8,Wire Harness,Initial_Rest,13769,855,12,0,0,6.30
9,Wire Harness,Rep1,30334,2472,1442,0,0,12.90
10,Wire Harness,Rep2,30773,3634,2512,0,0,19.97



### Full Raw HR and HRV Quality Check


In [71]:

# Store data-quality statistics from every processed CSV
raw_quality_records = []

for file in processed_files:
    task = file.relative_to(dataset_root).parts[0]

    # Read only HR and HRV in memory-efficient chunks
    for chunk in pd.read_csv(
        file,
        usecols=["HR", "HRV"],
        chunksize=50000
    ):
        # Convert values to numeric and treat infinity as invalid
        hr = pd.to_numeric(
            chunk["HR"], errors="coerce"
        ).replace([float("inf"), float("-inf")], float("nan"))

        hrv = pd.to_numeric(
            chunk["HRV"], errors="coerce"
        ).replace([float("inf"), float("-inf")], float("nan"))

        # Count potentially problematic readings
        raw_quality_records.append({
            "Task": task,
            "Total Rows": len(chunk),
            "HR <= 0": (hr <= 0).sum(),
            "HR > 220": (hr > 220).sum(),
            "HR Missing/Invalid": hr.isna().sum(),
            "HRV < 0": (hrv < 0).sum(),
            "HRV Absolute > 10000": (hrv.abs() > 10000).sum(),
            "HRV Missing/Invalid": hrv.isna().sum()
        })

# Combine results separately for each task
raw_quality_df = pd.DataFrame(raw_quality_records)

raw_quality_summary = (
    raw_quality_df.groupby("Task").sum()
)

display(raw_quality_summary)


,Total Rows,HR <= 0,HR > 220,HR Missing/Invalid,HRV < 0,HRV Absolute > 10000,HRV Missing/Invalid
Task,,,,,,,
Composite,180942,696,14,0,5377,6654,0
Wire Harness,182661,787,77,0,6381,8800,0



### HR and HRV Signal Quality Investigation


In [72]:

# Store HR and HRV quality statistics
quality_records = []

# Examine all processed CSV files
for file in processed_files:
    for chunk in pd.read_csv(
        file,
        usecols=["HR", "HRV", "ECGSQI"],
        chunksize=50000
    ):
        # Convert sensor readings to numeric values
        hr = pd.to_numeric(chunk["HR"], errors="coerce")
        hrv = pd.to_numeric(chunk["HRV"], errors="coerce")
        ecgsqi = pd.to_numeric(chunk["ECGSQI"], errors="coerce")

        # Separate readings by ECG signal quality
        quality_groups = {
            "ECGSQI = 0": ecgsqi == 0,
            "ECGSQI > 0": ecgsqi > 0,
            "Other/Missing ECGSQI": ecgsqi.isna() | (ecgsqi < 0)
        }

        # Count suspicious readings in each group
        for group, mask in quality_groups.items():
            quality_records.append({
                "ECG Quality": group,
                "Total Rows": mask.sum(),
                "HR <= 0": ((hr <= 0) & mask).sum(),
                "HR > 220": ((hr > 220) & mask).sum(),
                "HRV < 0": ((hrv < 0) & mask).sum(),
                "Absolute HRV > 10000": (
                    (hrv.abs() > 10000) & mask
                ).sum()
            })

# Combine the results
ecg_quality_summary = (
    pd.DataFrame(quality_records)
    .groupby("ECG Quality")
    .sum()
)

# Calculate suspicious HRV percentages
ecg_quality_summary["Negative HRV (%)"] = (
    ecg_quality_summary["HRV < 0"]
    / ecg_quality_summary["Total Rows"] * 100
).round(2)

ecg_quality_summary["Extreme HRV (%)"] = (
    ecg_quality_summary["Absolute HRV > 10000"]
    / ecg_quality_summary["Total Rows"] * 100
).round(2)

display(ecg_quality_summary)


,Total Rows,HR <= 0,HR > 220,HRV < 0,Absolute HRV > 10000,Negative HRV (%),Extreme HRV (%)
ECG Quality,,,,,,,
ECGSQI = 0,61623,331,5,2473,4769,4.01,7.74
ECGSQI > 0,301980,1152,86,9285,10685,3.07,3.54
Other/Missing ECGSQI,0,0,0,0,0,NaN,NaN



### HRV Quality by ECGSQI Level


In [73]:

# Store HRV quality statistics for different ECGSQI levels
quality_band_records = []

for file in processed_files:
    # Read HRV and ECGSQI in memory-efficient chunks
    for chunk in pd.read_csv(
        file,
        usecols=["HRV", "ECGSQI"],
        chunksize=50000
    ):
        hrv = pd.to_numeric(chunk["HRV"], errors="coerce")
        ecgsqi = pd.to_numeric(chunk["ECGSQI"], errors="coerce")

        # Define ECG signal-quality groups
        quality_bands = {
            "ECGSQI = 0": ecgsqi == 0,
            "0 < ECGSQI < 0.5": (
                (ecgsqi > 0) & (ecgsqi < 0.5)
            ),
            "ECGSQI >= 0.5": ecgsqi >= 0.5
        }

        # Count suspicious HRV readings in each group
        for band, mask in quality_bands.items():
            quality_band_records.append({
                "Quality Band": band,
                "Total Rows": mask.sum(),
                "Negative HRV": ((hrv < 0) & mask).sum(),
                "Extreme HRV": (
                    (hrv.abs() > 10000) & mask
                ).sum()
            })

# Combine statistics across all CSV files
quality_band_summary = (
    pd.DataFrame(quality_band_records)
    .groupby("Quality Band")
    .sum()
)

# Calculate percentages for each quality group
quality_band_summary["Negative HRV (%)"] = (
    quality_band_summary["Negative HRV"]
    / quality_band_summary["Total Rows"] * 100
).round(2)

quality_band_summary["Extreme HRV (%)"] = (
    quality_band_summary["Extreme HRV"]
    / quality_band_summary["Total Rows"] * 100
).round(2)

# Arrange groups in quality order
quality_band_summary = quality_band_summary.reindex([
    "ECGSQI = 0",
    "0 < ECGSQI < 0.5",
    "ECGSQI >= 0.5"
])

display(quality_band_summary)


,Total Rows,Negative HRV,Extreme HRV,Negative HRV (%),Extreme HRV (%)
Quality Band,,,,,
ECGSQI = 0,61623,2473,4769,4.01,7.74
0 < ECGSQI < 0.5,57188,1197,1516,2.09,2.65
ECGSQI >= 0.5,244792,8088,9169,3.30,3.75



### Extreme HRV Reading Investigation


In [74]:

# Track the largest absolute HRV value with ECGSQI >= 0.5
largest_hrv = -1
extreme_file = None
extreme_index = None

# Search all processed CSV files
for file in processed_files:
    data = pd.read_csv(
        file,
        usecols=["HRV", "ECGSQI"]
    )

    # Convert measurements to numeric values
    hrv = pd.to_numeric(data["HRV"], errors="coerce")
    ecgsqi = pd.to_numeric(data["ECGSQI"], errors="coerce")

    # Consider only readings with ECGSQI >= 0.5
    valid_quality_hrv = hrv.abs().where(ecgsqi >= 0.5)

    if valid_quality_hrv.notna().any():
        index = valid_quality_hrv.idxmax()
        value = valid_quality_hrv.loc[index]

        if value > largest_hrv:
            largest_hrv = value
            extreme_file = file
            extreme_index = index

# Inspect readings surrounding the extreme HRV value
if extreme_file is not None:
    extreme_data = pd.read_csv(
        extreme_file,
        usecols=["Time", "HR", "HRV", "ECGSQI"]
    )

    start = max(0, extreme_index - 5)
    end = min(len(extreme_data), extreme_index + 6)

    print("File:", extreme_file.name)
    print("Task:", extreme_file.relative_to(dataset_root).parts[0])
    print("Participant:", extreme_file.relative_to(dataset_root).parts[1])
    print("Extreme row index:", extreme_index)

    display(extreme_data.iloc[start:end])
else:
    print("No readings found with ECGSQI >= 0.5")


File: Initial_Rest_P039C039S011.csv
Task: Composite
Participant: P39
Extreme row index: 24


,Time,HR,HRV,ECGSQI
19,10500,1.145504,-1.390787e+11,0.000
20,11000,1.147211,-1.381716e+11,0.000
21,11500,1.159242,-1.372685e+11,0.000
22,12000,1.174966,-1.363694e+11,0.000
23,12500,1.190552,-1.354742e+11,0.000
24,13000,1.204155,-1.345829e+11,0.790
25,13500,1.213972,-1.336955e+11,0.770
26,14000,1.218202,-1.328121e+11,0.770
27,14500,1.215527,-1.319325e+11,0.756
28,15000,1.206839,-1.310569e+11,0.750



### Fatigue Label and Data Leakage Review


In [75]:

# Identify columns containing fatigue information
fatigue_columns_found = [
    column for column in reference_columns
    if "fatigue" in column.lower()
]

# Identify other columns that may contain outcome information
other_columns = [
    column for column in reference_columns
    if any(keyword in column.lower() for keyword in [
        "performance", "rating", "state"
    ])
]

print("Fatigue-related columns:")
for column in fatigue_columns_found:
    print("-", column)

print("\nOther columns requiring review:")
for column in other_columns:
    print("-", column)


Fatigue-related columns:
- Physical Fatigue - Initial
- Physical Fatigue - Final
- Mental Fatigue - Initial
- Mental Fatigue - Final

Other columns requiring review:
- Performance rating



### Dataset Understanding Summary


In [76]:

# Summarize the dataset structure
print("Unique participants:", scores_df["Participant"].nunique())
print("Experimental tasks:", scores_df["Task"].nunique())
print("Processed CSV files:", len(processed_files))
print("Total sensor rows:", full_quality_df["Total Rows"].sum())
print("Columns per CSV:", len(reference_columns))

# Summarize verified fatigue-label information
print("\nFatigue scale: 0–10")
print("Planned classes: Low, Moderate, High")
print("Fatigue labels: Session-level initial and final scores")

# Record preprocessing considerations
print("\nPreprocessing considerations:")
print("- Handle special HR codes (-2, -3)")
print("- Investigate or handle abnormal HRV values")
print("- Construct time-window fatigue labels")
print("- Exclude target and outcome columns from inputs")
print("- Use participant-independent train/test splits")


Unique participants: 38
Experimental tasks: 2
Processed CSV files: 469
Total sensor rows: 363603
Columns per CSV: 51

Fatigue scale: 0–10
Planned classes: Low, Moderate, High
Fatigue labels: Session-level initial and final scores

Preprocessing considerations:
- Handle special HR codes (-2, -3)
- Investigate or handle abnormal HRV values
- Construct time-window fatigue labels
- Exclude target and outcome columns from inputs
- Use participant-independent train/test splits
